# 지식산업센터 공실률 최소화 방안 - 전체 분석 파이프라인성남시 지식산업센터 데이터를 학습해 하남 교산지구의 격자별 공실률을 예측하고,공실률이 가장 낮을 것으로 예상되는 입지를 제안합니다.이 노트북은 원본 탐색적 분석(코드정리_분석.ipynb 등)을 `src/` 아래의 재사용 가능한함수/클래스로 리팩토링한 뒤, 그 함수들을 순서대로 호출하여 전체 흐름을 보여주는정리된 버전입니다. 실행하려면 `data/README.md`에 명시된 원본 데이터를`data/raw/`에 준비해야 합니다.

In [ ]:
import sysfrom pathlib import Pathsys.path.append(str(Path.cwd().parent))  # repo rootimport pandas as pdimport geopandas as gpdfrom src.config import RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, CENTER_COLUMN_RENAMEfrom src.preprocessing import vacancy, merchant, dataset_builderfrom src.grid import grid_features, buffer_featuresfrom src.analysis import edafrom src.modeling.pca_random_forest import VacancyRatePredictor

## 1. 원본 데이터 로드

In [ ]:
centers_raw = pd.read_csv(RAW_DATA_DIR / "7.성남시_지식산업센터.csv")centers = centers_raw.rename(columns=CENTER_COLUMN_RENAME)stores_raw = pd.read_csv(RAW_DATA_DIR / "성남_지산세_상가1.csv")stores = stores_raw.rename(columns={"klg_ids_ct_nm": "center_name"})building_ledger = pd.read_csv(RAW_DATA_DIR / "4.성남시_표제부.csv")transaction_price = pd.read_csv(RAW_DATA_DIR / "SN_지식산업센터_매매가.csv")

## 2. 공실률 산정건축물 전체 연면적 대비, 개별 상가(입주업체)의 실제 점유 면적 합을 비교해지식산업센터별 공실률을 계산합니다. 상가 데이터가 결측인 구간은평균 유닛 면적(avg_unit_area) 기반으로 보간합니다.

In [ ]:
centers = vacancy.drop_centers_missing_unit_count(centers, extra_drop=["SK V1"])centers = vacancy.compute_avg_unit_area(centers)stores = vacancy.interpolate_missing_store_area(stores, centers)centers = vacancy.compute_vacancy_rate(stores, centers)centers[["center_name", "vacancy_rate"]].describe()

## 3. 상가/건물 부가 데이터 결합

In [ ]:
active_stores = merchant.filter_active_stores(stores_raw)active_stores = merchant.add_store_age(active_stores)category_density = merchant.compute_category_density(centers, active_stores, radius_m=250)centers = dataset_builder.merge_elevator_and_parking(centers, building_ledger)centers = dataset_builder.merge_transaction_price(centers, transaction_price)

## 4. 100m 격자 데이터셋 구성유동인구·거주인구·카드매출·상권·대중교통·공시지가를 100m 격자 단위로 결합합니다.

In [ ]:
grid = grid_features.load_grid(str(RAW_DATA_DIR / "15.성남시_격자(100M).geojson"))floating_pop = pd.read_csv(RAW_DATA_DIR / "성남시_유동인구.csv")card_sales = pd.read_csv(RAW_DATA_DIR / "성남_카드매출.csv")resident_pop = pd.read_csv(RAW_DATA_DIR / "1.성남시_거주인구.csv")district = pd.read_csv(RAW_DATA_DIR / "2.성남시_상권정보.csv")subway_stations = pd.read_csv(RAW_DATA_DIR / "10.성남시_지하철역.csv")bus_stops = pd.read_csv(RAW_DATA_DIR / "9.성남시_버스정류장.csv")land_price = pd.read_csv(RAW_DATA_DIR / "8.성남시_개별공시지가.csv")grid_dataset = grid_features.build_grid_dataset(    grid=grid,    floating_pop_df=floating_pop,    card_sales_df=card_sales,    resident_pop_df=resident_pop,    district_df=district,    subway_df=subway_stations,    bus_df=bus_stops,    land_price_df=land_price,    resident_pop_year=2023,)grid_dataset.to_csv(INTERIM_DATA_DIR / "seongnam_grid_features.csv", index=False)grid_dataset.head()

## 5. Buffer 기반 센터별 특성 결합각 지식산업센터를 기준으로 상권(500m) / 거주인구(100m) / 지하철(2000m) buffer를적용해 주변 환경 데이터를 집계하고, 가장 가까운 버스정류장·지하철역까지의거리를 추가합니다.

In [ ]:
buffer_df = buffer_features.build_buffer_dataset(    centers_df=centers,    commercial_grid=grid_dataset,    population_grid=grid_dataset,    subway_grid=grid_dataset,    bus_stops_df=bus_stops,    subway_stations_df=subway_stations,)buffer_df.to_csv(PROCESSED_DATA_DIR / "seongnam_buffer_features.csv", index=False)buffer_df.shape

## 6. EDA: 상관관계 / 다중공선성 확인

In [ ]:
eda.plot_correlation_heatmap(buffer_df)

In [ ]:
vif_table = eda.compute_vif(buffer_df.select_dtypes(include="number"))vif_table.head(15)

## 7. 모델링: PCA + RandomForestVIF가 10을 크게 넘는 변수가 다수 확인되어(다중공선성), 원본 특성을 그대로 회귀에사용하지 않고 PCA로 차원을 축소한 뒤 RandomForest로 학습합니다.(지리가중회귀 / 군집분석 / 다중회귀 / 로지스틱회귀 대비 test MSE가 가장 낮았습니다 - README 참고)

In [ ]:
predictor = VacancyRatePredictor()predictor.fit(buffer_df)print(predictor.test_metrics_)predictor.feature_importance()

In [ ]:
predictor.pca_loadings()

## 8. 하남 교산지구 격자 공실률 예측동일하게 학습된 scaler / PCA / RandomForest를 하남 교산지구 격자 데이터셋에그대로 적용하여 격자별 예상 공실률을 산출합니다.

In [ ]:
hanam_grid_raw = gpd.read_file(RAW_DATA_DIR / "20.하남시_격자(100M).geojson")hanam_grid = hanam_grid_raw.rename(columns={"gid": "grid_id"})hanam_grid["predicted_vacancy_rate"] = predictor.predict(hanam_grid)hanam_grid[["grid_id", "predicted_vacancy_rate", "geometry"]].to_csv(    PROCESSED_DATA_DIR / "hanam_gyosan_predicted_vacancy.csv", index=False)hanam_grid.sort_values("predicted_vacancy_rate").head(10)

## 9. 결론예측된 격자별 공실률을 상권·교통이 발달한 지역과 겹쳐 시각화한 결과, 대로변에낮은 공실률이 예상되는 격자가 밀집되어 있음을 확인했습니다. 이를 바탕으로**샘재입구 사거리 부근**을 최적 입지로 제안합니다 (예상 공실률 0.7).자세한 내용과 발표 슬라이드는 `presentation/` 폴더 및 프로젝트 루트 `README.md`를참고하세요.